In [ ]:
!pip install oracledb pandas sqlalchemy --quiet

import oracledb
import pandas as pd
from sqlalchemy import create_engine

print('Libraries loaded')

In [ ]:
USERNAME = ""
PASSWORD = ""
DSN      = "tcps://db.freesql.com:2484/23ai_34ui2"

engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={"user": USERNAME, "password": PASSWORD, "dsn": DSN}
)

pd.read_sql("SELECT COUNT(*) AS n FROM tickets", engine)

In [ ]:
tickets_df     = pd.read_sql("SELECT * FROM tickets ORDER BY ticket_id", engine)
assignments_df = pd.read_sql("SELECT * FROM ticket_assignments ORDER BY ticket_id, valid_from", engine)

print(f"{len(tickets_df)} tickets, {len(assignments_df)} assignment records")
display(tickets_df)
display(assignments_df)

In [ ]:
# Agent assigned at creation time
created = tickets_df[['ticket_id', 'created_at', 'status', 'priority']].merge(
    assignments_df[['ticket_id', 'assigned_to', 'valid_from', 'valid_to']],
    on='ticket_id'
)
created = created[
    (created['created_at'] >= created['valid_from']) &
    (created['valid_to'].isna() | (created['created_at'] < created['valid_to']))
].copy()
created['date_key'] = created['created_at'].dt.strftime('%Y%m%d').astype(int)

# Agent assigned at resolution time
resolved = tickets_df[tickets_df['resolved_at'].notna()][['ticket_id', 'resolved_at', 'status', 'priority']].merge(
    assignments_df[['ticket_id', 'assigned_to', 'valid_from', 'valid_to']],
    on='ticket_id'
)
resolved = resolved[
    (resolved['resolved_at'] >= resolved['valid_from']) &
    (resolved['valid_to'].isna() | (resolved['resolved_at'] < resolved['valid_to']))
].copy()
resolved['date_key'] = resolved['resolved_at'].dt.strftime('%Y%m%d').astype(int)

print(f"Creation matches: {len(created)}, Resolution matches: {len(resolved)}")

In [ ]:
created_grp  = created.groupby(['date_key', 'assigned_to', 'status', 'priority']).size().reset_index(name='tickets_created')
resolved_grp = resolved.groupby(['date_key', 'assigned_to', 'status', 'priority']).size().reset_index(name='tickets_resolved')

fact = created_grp.merge(resolved_grp, on=['date_key', 'assigned_to', 'status', 'priority'], how='outer').fillna(0)
fact['tickets_created']  = fact['tickets_created'].astype(int)
fact['tickets_resolved'] = fact['tickets_resolved'].astype(int)
fact = fact.rename(columns={'assigned_to': 'agent_key'})

print(f"Fact rows to load: {len(fact)}")
display(fact)

In [ ]:
merge_sql = """
    MERGE INTO fact_ticket_daily f
    USING (SELECT :1 AS date_key, :2 AS agent_key, :3 AS status,
                  :4 AS priority, :5 AS tickets_created, :6 AS tickets_resolved
             FROM dual) src
    ON (f.date_key  = src.date_key
    AND f.agent_key = src.agent_key
    AND f.status    = src.status
    AND f.priority  = src.priority)
    WHEN NOT MATCHED THEN INSERT
        (date_key, agent_key, status, priority, tickets_created, tickets_resolved)
        VALUES (src.date_key, src.agent_key, src.status, src.priority,
                src.tickets_created, src.tickets_resolved)
    WHEN MATCHED THEN UPDATE SET
        tickets_created  = src.tickets_created,
        tickets_resolved = src.tickets_resolved
"""

conn   = engine.raw_connection()
cursor = conn.cursor()

for _, row in fact.iterrows():
    cursor.execute(merge_sql, [
        int(row['date_key']), int(row['agent_key']),
        row['status'], row['priority'],
        int(row['tickets_created']), int(row['tickets_resolved'])
    ])

conn.commit()
cursor.close()
conn.close()
print(f"Loaded {len(fact)} rows into fact_ticket_daily")